In [1]:
# ============================================================
# FULL REVISED ANALYSIS SCRIPT
# Fixes: Bootstrapped Mediation, CIs, Standardized Beta,
#        Harman's CMB, OLS Assumptions, HTMT, MCAR Test,
#        Holm-Bonferroni Correction
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from itertools import combinations

# ============================================================
# 1. LOAD AND CLEAN DATA
# ============================================================
file_path = './GRUMBLING BEHAVIOR (STEM) (Responses).xlsx'
df = pd.read_excel(file_path)

likert_map = {
    'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3,
    'Agree': 4, 'Strongly agree': 5
}

cols = {
    'SC':  [df.columns[4], df.columns[5]],
    'SB':  [df.columns[6], df.columns[7]],
    'WO':  [df.columns[8], df.columns[9]],
    'RA':  [df.columns[10], df.columns[11]],
    'WHC': [df.columns[12]],
    'GG':  [df.columns[13], df.columns[14], df.columns[15]],
    'W':   [df.columns[16], df.columns[17]],
    'LTA': [df.columns[18]]
}

df_path = pd.DataFrame()
for construct, item_list in cols.items():
    temp_df = df[item_list].replace(likert_map)
    df_path[construct] = temp_df.mean(axis=1)

# ============================================================
# 2. GENDER MCAR TEST (Fix M1)
# ============================================================
print("=" * 65)
print("SECTION 1: GENDER MISSING DATA — MCAR TEST")
print("=" * 65)

gender_col = df.columns[3]  # adjust index if needed
df['gender_missing'] = df[gender_col].isna().astype(int)

# Map year of study to numeric if needed
year_col = df.columns[2]  # adjust index if needed
df['year_num'] = pd.Categorical(df[year_col]).codes

# Chi-square: gender missingness vs year of study
ct = pd.crosstab(df['gender_missing'], df['year_num'])
chi2, p_val, dof, _ = stats.chi2_contingency(ct)
print(f"Chi-square (gender missing vs year): χ²={chi2:.3f}, df={dof}, p={p_val:.3f}")
if p_val > 0.05:
    print("→ Non-significant: gender non-response appears MCAR (not associated with year of study).")
else:
    print("→ Significant: gender non-response IS associated with year of study. Report as MAR, not MCAR.")

# ============================================================
# 3. HARMAN'S SINGLE-FACTOR TEST — Common Method Bias (Fix A4)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 2: HARMAN'S SINGLE-FACTOR TEST (Common Method Bias)")
print("=" * 65)

# Collect all individual items
all_items = []
for construct, item_list in cols.items():
    for col in item_list:
        all_items.append(col)

df_items = df[all_items].replace(likert_map).dropna()

# Standardize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_items)

# PCA — force 1 factor
from sklearn.decomposition import PCA
pca1 = PCA(n_components=1)
pca1.fit(X_scaled)
variance_1factor = pca1.explained_variance_ratio_[0] * 100

pca_all = PCA()
pca_all.fit(X_scaled)
total_variance = sum(pca_all.explained_variance_ratio_) * 100

print(f"Single factor explains: {variance_1factor:.2f}% of total variance")
print(f"Threshold: 50%")
if variance_1factor < 50:
    print(f"→ PASS: CMB is unlikely to severely distort findings (< 50% threshold).")
    print(f"  Report: 'Harman's single-factor test: single factor accounted for {variance_1factor:.1f}% of variance.'")
else:
    print(f"→ CAUTION: Single factor > 50%. CMB may be a concern. Acknowledge in limitations.")

# ============================================================
# 4. HTMT DISCRIMINANT VALIDITY (Fix A3)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 3: HTMT DISCRIMINANT VALIDITY")
print("=" * 65)

def compute_htmt(df_items_dict, df_raw):
    """Compute HTMT between all construct pairs."""
    construct_names = list(df_items_dict.keys())
    results = []
    for c1, c2 in combinations(construct_names, 2):
        items1 = df_raw[df_items_dict[c1]].replace(likert_map)
        items2 = df_raw[df_items_dict[c2]].replace(likert_map)
        # Average inter-construct correlations
        cross_corrs = []
        for i in df_items_dict[c1]:
            for j in df_items_dict[c2]:
                r = df_raw[i].replace(likert_map).corr(df_raw[j].replace(likert_map))
                cross_corrs.append(abs(r))
        # Average within-construct correlations (geometric mean approach)
        within1 = []
        for i, j in combinations(df_items_dict[c1], 2):
            r = df_raw[i].replace(likert_map).corr(df_raw[j].replace(likert_map))
            within1.append(abs(r))
        within2 = []
        for i, j in combinations(df_items_dict[c2], 2):
            r = df_raw[i].replace(likert_map).corr(df_raw[j].replace(likert_map))
            within2.append(abs(r))
        avg_cross = np.mean(cross_corrs)
        avg_w1 = np.mean(within1) if within1 else 1.0
        avg_w2 = np.mean(within2) if within2 else 1.0
        htmt = avg_cross / np.sqrt(avg_w1 * avg_w2)
        results.append({'Construct 1': c1, 'Construct 2': c2, 'HTMT': round(htmt, 3)})
    return pd.DataFrame(results)

# Only multi-item constructs for HTMT
cols_multi = {k: v for k, v in cols.items() if len(v) > 1}
htmt_df = compute_htmt(cols_multi, df)
htmt_df['Status'] = htmt_df['HTMT'].apply(lambda x: 'OK (<0.90)' if x < 0.90 else 'FAIL (≥0.90)')
print(htmt_df.to_string(index=False))
print("\n→ All HTMT values should be < 0.90 for discriminant validity.")

# ============================================================
# 5. PATH ANALYSIS WITH CIs AND STANDARDIZED BETA (Fixes R1, R2)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 4: PATH ANALYSIS — B, β, SE, 95% CI, p-value")
print("=" * 65)

def standardize(series):
    return (series - series.mean()) / series.std()

df_std = df_path.apply(standardize)

def report_model(model, model_std, label):
    print(f"\n--- {label} ---")
    params = model.params.drop('Intercept', errors='ignore')
    conf = model.conf_int().drop('Intercept', errors='ignore')
    bse = model.bse.drop('Intercept', errors='ignore')
    pvals = model.pvalues.drop('Intercept', errors='ignore')
    beta = model_std.params.drop('Intercept', errors='ignore')

    rows = []
    for var in params.index:
        rows.append({
            'Predictor': var,
            'B': round(params[var], 3),
            'β (std)': round(beta.get(var, np.nan), 3),
            'SE': round(bse[var], 3),
            '95% CI Lower': round(conf.loc[var, 0], 3),
            '95% CI Upper': round(conf.loc[var, 1], 3),
            'p-value': round(pvals[var], 4)
        })
    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    print(f"R² = {model.rsquared:.3f}, Adj. R² = {model.rsquared_adj:.3f}, "
          f"F = {model.fvalue:.2f}, p = {model.f_pvalue:.4f}, N = {int(model.nobs)}")

# Model 1: WHC ~ WO + SC
m1 = smf.ols('WHC ~ WO + SC', data=df_path).fit()
m1s = smf.ols('WHC ~ WO + SC', data=df_std).fit()
report_model(m1, m1s, "MODEL 1: Predictors of Work-Home Conflict")

# Model 2: W ~ SB
m2 = smf.ols('W ~ SB', data=df_path).fit()
m2s = smf.ols('W ~ SB', data=df_std).fit()
report_model(m2, m2s, "MODEL 2: Predictors of Intrinsic Willingness")

# Model 3: GG ~ all
m3 = smf.ols('GG ~ RA + WHC + SC + SB + W + LTA + WO', data=df_path).fit()
m3s = smf.ols('GG ~ RA + WHC + SC + SB + W + LTA + WO', data=df_std).fit()
report_model(m3, m3s, "MODEL 3: Predictors of General Resistance")

# ============================================================
# 6. OLS ASSUMPTION CHECKS (Fix A2)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 5: OLS ASSUMPTION CHECKS")
print("=" * 65)

for model, label in [(m1, "Model 1 (WHC)"), (m2, "Model 2 (W)"), (m3, "Model 3 (GG)")]:
    residuals = model.resid
    fitted = model.fittedvalues

    # Shapiro-Wilk normality of residuals
    sw_stat, sw_p = stats.shapiro(residuals)

    # Breusch-Pagan heteroscedasticity
    from statsmodels.stats.diagnostic import het_breuschpagan
    bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(residuals, model.model.exog)

    print(f"\n{label}:")
    print(f"  Shapiro-Wilk (normality):       W={sw_stat:.4f}, p={sw_p:.4f} "
          f"{'→ Normal ✓' if sw_p > 0.05 else '→ Non-normal ✗ (report in limitations)'}")
    print(f"  Breusch-Pagan (homoscedasticity): LM={bp_lm:.4f}, p={bp_p:.4f} "
          f"{'→ Homoscedastic ✓' if bp_p > 0.05 else '→ Heteroscedastic ✗ (use robust SEs)'}")

# ============================================================
# 7. VIF (existing, kept for completeness)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 6: MULTICOLLINEARITY CHECK (VIF)")
print("=" * 65)

X = df_path[['RA', 'WHC', 'SC', 'SB', 'W', 'LTA', 'WO']]
X_const = sm.add_constant(X)
vif = pd.DataFrame()
vif["Variable"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
print(vif[vif["Variable"] != "const"].to_string(index=False))

# ============================================================
# 8. BOOTSTRAPPED MEDIATION — WO → WHC → GG (Fix A1)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 7: BOOTSTRAPPED MEDIATION ANALYSIS")
print("  WO → WHC → GG  (Hayes PROCESS Model 4 equivalent)")
print("=" * 65)

np.random.seed(42)
N_BOOT = 5000
n = len(df_path)

indirect_effects = []

for _ in range(N_BOOT):
    sample = df_path.sample(n=n, replace=True)

    # Path a: WO → WHC (controlling SC)
    m_a = smf.ols('WHC ~ WO + SC', data=sample).fit()
    a = m_a.params['WO']

    # Path b: WHC → GG (controlling all other predictors)
    m_b = smf.ols('GG ~ RA + WHC + SC + SB + W + LTA + WO', data=sample).fit()
    b = m_b.params['WHC']

    indirect_effects.append(a * b)

indirect_effects = np.array(indirect_effects)
ci_lower = np.percentile(indirect_effects, 2.5)
ci_upper = np.percentile(indirect_effects, 97.5)
point_estimate = indirect_effects.mean()

# Point estimate from original data
a_orig = m1.params['WO']
b_orig = m3.params['WHC']
indirect_orig = a_orig * b_orig

print(f"\nPath a  (WO → WHC):  B = {a_orig:.4f}")
print(f"Path b  (WHC → GG):  B = {b_orig:.4f}")
print(f"Indirect effect (a×b): {indirect_orig:.4f}")
print(f"Bootstrap mean:        {point_estimate:.4f}")
print(f"95% Bootstrap CI:      [{ci_lower:.4f}, {ci_upper:.4f}]  (k={N_BOOT} resamples)")

if ci_lower > 0 or ci_upper < 0:
    print("→ CI excludes zero: MEDIATION IS SUPPORTED ✓")
    print("  Report: 'The indirect effect of Work Overload on General Resistance")
    print(f"  via Work-Home Conflict was {indirect_orig:.3f} (95% CI [{ci_lower:.3f}, {ci_upper:.3f}]),")
    print(f"  based on {N_BOOT} bootstrap resamples, confirming mediation.'")
else:
    print("→ CI includes zero: Mediation is NOT supported at 95% confidence.")
    print("  You must report this honestly and revise H2.")

# ============================================================
# 9. HOLM-BONFERRONI CORRECTION (Fix R4)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 8: HOLM-BONFERRONI MULTIPLE TESTING CORRECTION")
print("=" * 65)

hypotheses = {
    'H1: RA → GG':        m3.pvalues['RA'],
    'H2: WO→WHC→GG (indirect)': None,   # use bootstrap CI result
    'H3: SC → WHC':       m1.pvalues['SC'],
    'H4: SB → W':         m2.pvalues['SB'],
    'H5: W → GG':         m3.pvalues['W'],
}

# Holm-Bonferroni on the 4 directly tested p-values
direct_hyp = {k: v for k, v in hypotheses.items() if v is not None}
sorted_hyp = sorted(direct_hyp.items(), key=lambda x: x[1])
m_total = len(sorted_hyp)

print(f"\n{'Hypothesis':<25} {'p-value':>10} {'Holm threshold':>16} {'Decision':>12}")
print("-" * 68)
for rank, (hyp, p) in enumerate(sorted_hyp, start=1):
    threshold = 0.05 / (m_total - rank + 1)
    decision = "Significant ✓" if p < threshold else "NOT significant ✗"
    print(f"{hyp:<25} {p:>10.4f} {threshold:>16.4f} {decision:>12}")

print("\nH2 (mediation): assessed via bootstrap CI above (not a p-value test).")
print("→ If CI excludes zero, H2 is supported regardless of Holm correction.")

print("\n" + "=" * 65)
print("ALL ANALYSES COMPLETE.")
print("Copy the values above directly into your paper.")
print("=" * 65)

C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = df[item_list].replace(likert_map)
C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = df[item_list].replace(likert_map)
C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result

SECTION 1: GENDER MISSING DATA — MCAR TEST
Chi-square (gender missing vs year): χ²=0.000, df=0, p=1.000
→ Non-significant: gender non-response appears MCAR (not associated with year of study).

SECTION 2: HARMAN'S SINGLE-FACTOR TEST (Common Method Bias)
Single factor explains: 30.15% of total variance
Threshold: 50%
→ PASS: CMB is unlikely to severely distort findings (< 50% threshold).
  Report: 'Harman's single-factor test: single factor accounted for 30.2% of variance.'

SECTION 3: HTMT DISCRIMINANT VALIDITY
Construct 1 Construct 2  HTMT     Status
         SC          SB 0.179 OK (<0.90)
         SC          WO 0.612 OK (<0.90)
         SC          RA 0.494 OK (<0.90)
         SC          GG 0.479 OK (<0.90)
         SC           W 0.136 OK (<0.90)
         SB          WO 0.132 OK (<0.90)
         SB          RA 0.306 OK (<0.90)
         SB          GG 0.352 OK (<0.90)
         SB           W 0.505 OK (<0.90)
         WO          RA 0.550 OK (<0.90)
         WO          GG 0.536 OK

C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:116: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  items1 = df_raw[df_items_dict[c1]].replace(likert_map)
C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:117: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  items2 = df_raw[df_items_dict[c2]].replace(likert_map)
C:\Users\Kafi\AppData\Local\Temp\ipykernel_14264\2915485089.py:122: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior

Variable      VIF
      RA 1.374327
     WHC 1.502752
      SC 1.342255
      SB 1.235589
       W 1.439515
     LTA 1.345081
      WO 1.532926

SECTION 7: BOOTSTRAPPED MEDIATION ANALYSIS
  WO → WHC → GG  (Hayes PROCESS Model 4 equivalent)

Path a  (WO → WHC):  B = 0.4418
Path b  (WHC → GG):  B = 0.1949
Indirect effect (a×b): 0.0861
Bootstrap mean:        0.0878
95% Bootstrap CI:      [0.0214, 0.1633]  (k=5000 resamples)
→ CI excludes zero: MEDIATION IS SUPPORTED ✓
  Report: 'The indirect effect of Work Overload on General Resistance
  via Work-Home Conflict was 0.086 (95% CI [0.021, 0.163]),
  based on 5000 bootstrap resamples, confirming mediation.'

SECTION 8: HOLM-BONFERRONI MULTIPLE TESTING CORRECTION

Hypothesis                   p-value   Holm threshold     Decision
--------------------------------------------------------------------
H1: RA → GG                   0.0000           0.0125 Significant ✓
H4: SB → W                    0.0000           0.0167 Significant ✓
H3: SC → WH